## Note 4.1: Examples of float64 bit calculation

### What you will learn

In this notebook, we:
- get familiar with float64 bit calculation (addition and multiplication) through numerous examples

### Examples to explain
#### Addition

In [1]:
print(0.001 + 0.002)
print(0.01 + 0.02)
print(0.005 + 0.02)
print(0.1 + 0.7)
print(5.8 + 53.62)

0.003
0.03
0.025
0.7999999999999999
59.419999999999995


(The first two examples are in contrast with a famous one: 0.1+0.2=0.30000000000000004)
#### Multiplication

In [2]:
print(1.3 * 3.2)
print(0.1 * 0.7)
print(1e2 * 5.4321)

4.16
0.06999999999999999
543.21


In [3]:
print(f"{0.3:.70f}")

0.2999999999999999888977697537484345957636833190917968750000000000000000


Let us next think about addition.  When we tell a computer to perform addition of two real numbers, $r_1+r_2$, the computer first maps $r_1$ and $r_2$ to their corresponding discretized numbers, $\bar{r}_1$ and $\bar{r}_2$, and then makes a sum, $\bar{r}_1 + \bar{r}_2$. The result is also mapped to the nearby discretized number, and the final result is $\overline{\bar{r}_1 + \bar{r}_2}$.  

An important point is that it can happen that $\overline{\bar{r}_1 + \bar{r}_2} \neq \overline{r_1 + r_2}$.  Namely, when the rounding errors, $r_1-\bar{r}_1$ and $r_2-\bar{r}_2$, are unfortunately large, $\bar{r}_1 + \bar{r}_2$ can happen to reside in a bin different from the bin where $r_1+r_2$ resides.  

The example $0.1+0.2=0.30000000000000004$ is exactly the case: $\overline{0.1}+\overline{0.2}$ is rounded to a discretized number different from which 0.3 is mapped to (compare to the result of print(f"{0.3:.70f}") above). 

In [4]:
print(f"{0.1+0.2:.70f}")

0.3000000000000000444089209850062616169452667236328125000000000000000000


On the other hand, when the rounding errors are fortunately small, then $\overline{\bar{r}_1 + \bar{r}_2} = \overline{r_1 + r_2}$ holds.  The example $0.11 + 0.19 = 0.3$ is this case:

In [5]:
print(f"{0.11+0.19:.70f}")

0.2999999999999999888977697537484345957636833190917968750000000000000000


Compare the rounding errors in the two cases: 

In [6]:
print(f"{0.1:.70f}")
print(f"{0.11:.70f}")
print(f"{0.2:.70f}")
print(f"{0.19:.70f}")

0.1000000000000000055511151231257827021181583404541015625000000000000000
0.1100000000000000005551115123125782702118158340454101562500000000000000
0.2000000000000000111022302462515654042363166809082031250000000000000000
0.1900000000000000022204460492503130808472633361816406250000000000000000


We see that the rounding errors are all the same sign (negative), and they tend to add up in the calculation of addition.  But the rounding errors of $0.11$ and $0.19$ are smaller than $0.1$ and $0.2$, respectively, the addition $0.11+0.19$ equals to $0.3$ while $0.1+0.2$ does not. 

Now we have grasped the essene.   The remaining part of this notebook is an appendix: it is a detailed and explicit demonstration of the bit calculation of the examples for interested readers.  

### Helper functions 
Below are functions to perform conversion between decimal representation and its float64 binary representation.

In [7]:
import numpy as np
from functools import partial

def decompose_float64(d: float) -> tuple[str, str, str]:  
    """
    Given a float64 number, this function returns its sign, exponent, fraction part as strings
    Args:
        d (float): Input floating-point number.
    Returns:
        tuple[str, str, str]:
            A tuple containing:
            - sign_str (str): The sign bit 
            - exponent_str (str): The 11-bit exponent
            - fraction_str (str): The 52-bit fraction
    """
    x = np.float64(d)
    x_int64 = x.view(np.int64)  # reinterpret the bits of x as those of int64 and returns the corresponding int64 number 
    binary_str = str(f"{x_int64:064b}")  # returns the string of binary representation of x_int64
    sign_str = binary_str[0]
    exponent_str = binary_str[1:12]
    fraction_str = binary_str[12:]
    return sign_str, exponent_str, fraction_str

def compose_float64(sign_str: str, exponent_str: str, fraction_str: str) -> float:
    """
    This is the "inverse" of decompose_float64
    """
    assert len(sign_str)==1 and len(exponent_str)==11 and len(fraction_str)==52
    assert set(sign_str)<= {'0', '1'} and set(exponent_str)<= {'0', '1'} and set(fraction_str)<= {'0', '1'}

    binary_str = sign_str + exponent_str + fraction_str
    x = int(binary_str, 2)   # interpret binary_str as binary representation of an integer and returns that integer
    x_int64 = np.int64(x)
    x_float64 = x_int64.view(np.float64)  # reinterpret the bits of x_int64 as those of float64 and returns the float64 number

    return x_float64.item()

# floating point addition

According to [https://www.sciencedirect.com/topics/computer-science/floating-point-addition](https://www.sciencedirect.com/topics/computer-science/floating-point-addition), the rule of floating point addition is as follows:


1. Extract exponent and fraction bits.  
2. Prepend leading 1 to form the mantissa.  
3. Compare exponents.  
4. Shift smaller mantissa if necessary.  
5. Add mantissas.  
6. Normalize mantissa and adjust exponent if necessary.  
7. Round result.  
8. Assemble exponent and fraction back into floating-point number.  

(mantissa is the same meaning as the word "fraction" we use here.)  

Let us see the examples one by one.  

In the explanation below, we represent float64 as {exponent part}_{fraction part}. We omit sign part since we treat only positive numbers.
### Explanation of 0.001 + 0.002 = 0.003

In [8]:
print(decompose_float64(0.001))
print(decompose_float64(0.002))

('0', '01111110101', '0000011000100100110111010010111100011010100111111100')
('0', '01111110110', '0000011000100100110111010010111100011010100111111100')


Name the former as A and the latter as B.  We connect the exponent and the fraction by '__':  
A = 01111110101___0000011000100100110111010010111100011010100111111100  
B = 01111110110___0000011000100100110111010010111100011010100111111100  
Prepend "1." to the fraction part, the result of which we call "1+f part":  
A = 01111110101_1.0000011000100100110111010010111100011010100111111100  
B = 01111110110_1.0000011000100100110111010010111100011010100111111100  
Make the exponent of A as the same as that of B, and shift the 1+f part to the right:  
A = 01111110110_0.1000001100010010011011101001011110001101010011111110  
B = 01111110110_1.0000011000100100110111010010111100011010100111111100  
Add the 1+f parts, and call the result C:   
C = 01111110110_1.1000100100110111010010111100011010100111111011111010  
Delete "1.":  
C = 01111110110___1000100100110111010010111100011010100111111011111010

Convert this float64 bit representation to decimal representation: 

In [9]:
compose_float64("0", "01111110110", "1000100100110111010010111100011010100111111011111010")

0.003

### Explanation of 0.01 + 0.02 = 0.03

In [10]:
print(decompose_float64(0.01))
print(decompose_float64(0.02))

('0', '01111111000', '0100011110101110000101000111101011100001010001111011')
('0', '01111111001', '0100011110101110000101000111101011100001010001111011')


Name the former as A and the latter as B.  We connect the exponent and the fraction by '__':  
A = 01111111000___0100011110101110000101000111101011100001010001111011  
B = 01111111001___0100011110101110000101000111101011100001010001111011  
Prepend "1." to the fraction part, the result of which we call "1+f part":   
A = 01111111000_1.0100011110101110000101000111101011100001010001111011  
B = 01111111001_1.0100011110101110000101000111101011100001010001111011  
Make the exponent of A as the same as that of B, and shift the 1+f part to the right:  
A = 01111111001_0.10100011110101110000101000111101011100001010001111011  
B = 01111111001_1.0100011110101110000101000111101011100001010001111011  
Add the 1+f parts, and call the result C:   
C = 01111111001_1.11101011100001010001111010111000010100011110101110001  
Round the fraction part:  
C = 01111111001_1.1110101110000101000111101011100001010001111010111000  
Delete "1.":  
C = 01111111001___1110101110000101000111101011100001010001111010111000  

**Remark:  
In the rounding process, "round to even" rule is applied.  That is, in the intermediate result before the rounding,  
C = 01111111001_1.11101011100001010001111010111000010100011110101110001,   
the 53rd bit after the binary point is '1', and the bits after that are regarded as '000...'.  In such a 'tie' case, the rounding is performed so that the 52nd bit after the binary point becomes 0.  Compare this to the next example, 0.005 + 0.02 = 0.025, which is not such a tie case when rounding.**  

Convert the resultant float64 bit representation to decimal representation: 

In [11]:
compose_float64("0", "01111111001", "1110101110000101000111101011100001010001111010111000")

0.03

## Explanation of 0.005 + 0.02 = 0.025

In [12]:
print(decompose_float64(0.005))
print(decompose_float64(0.02))

('0', '01111110111', '0100011110101110000101000111101011100001010001111011')
('0', '01111111001', '0100011110101110000101000111101011100001010001111011')


Name the former as A and the latter as B. We connect the exponent and the fraction by '__':  
A = 01111110111___0100011110101110000101000111101011100001010001111011  
B = 01111111001___0100011110101110000101000111101011100001010001111011  
Prepend "1." to the fraction part, the result of which we call "1+f part":  
A = 01111110111_1.0100011110101110000101000111101011100001010001111011  
B = 01111111001_1.0100011110101110000101000111101011100001010001111011  
Make the exponent of A as the same as that of B, and shift the 1+f part to the right 2 times:  
A = 01111111001_0.010100011110101110000101000111101011100001010001111011  
B = 01111111001_1.0100011110101110000101000111101011100001010001111011  
Add the 1+f parts, and call the result C:  
C = 01111111001_1.100110011001100110011001100110011001100110011001100111    
Round the fraction part:   
C = 01111111001_1.1001100110011001100110011001100110011001100110011010  
Delete "1.":  
C = 01111111001_1001100110011001100110011001100110011001100110011010 

Convert the resultant float64 bit representation to decimal representation:

In [13]:
compose_float64("0", "01111111001", "1001100110011001100110011001100110011001100110011010")

0.025

### Explanation of 0.1 + 0.7 = 0.7999999999999999

In [14]:
print(decompose_float64(0.1))
print(decompose_float64(0.7))

('0', '01111111011', '1001100110011001100110011001100110011001100110011010')
('0', '01111111110', '0110011001100110011001100110011001100110011001100110')


Name the former as A and the latter as B.  We connect the exponent and the fraction by '__':  
A = 01111111011___1001100110011001100110011001100110011001100110011010  
B = 01111111110___0110011001100110011001100110011001100110011001100110  
Prepend "1." to the fraction part, the result of which we call "1+f part":  
A = 01111111011_1.1001100110011001100110011001100110011001100110011010  
B = 01111111110_1.0110011001100110011001100110011001100110011001100110  
Make the exponent of A as the same as that of B, and shift the 1+f part to the right 3 times:  
A = 01111111110_0.0011001100110011001100110011001100110011001100110011010  
B = 01111111110_1.0110011001100110011001100110011001100110011001100110  
Add the 1+f parts, and call the result C:   
C = 01111111110_1.1001100110011001100110011001100110011001100110011001010  
Round the fraction part of C:  
C = 01111111110_1.1001100110011001100110011001100110011001100110011001  
Delete "1.":  
C = 01111111110___1001100110011001100110011001100110011001100110011001  

Convert the resultant float64 bit representation to decimal representation:

In [15]:
compose_float64("0", "01111111110", "1001100110011001100110011001100110011001100110011001")

0.7999999999999999

### Explanation of 5.8 + 53.62 = 59.419999999999995

In [16]:
print(decompose_float64(5.8))
print(decompose_float64(53.62))

('0', '10000000001', '0111001100110011001100110011001100110011001100110011')
('0', '10000000100', '1010110011110101110000101000111101011100001010001111')


Name the former as A and the latter as B.  We connect the exponent and the fraction by '__':  
A = 10000000001___0111001100110011001100110011001100110011001100110011  
B = 10000000100___1010110011110101110000101000111101011100001010001111  
Prepend "1." to the fraction part, the result of which we call "1+f part":  
A = 10000000001_1.0111001100110011001100110011001100110011001100110011  
B = 10000000100_1.1010110011110101110000101000111101011100001010001111  
Make the exponent of A as the same as that of B, and shift the 1+f part to the right 3 times:  
A = 10000000100_0.0010111001100110011001100110011001100110011001100110011  
B = 10000000100_1.1010110011110101110000101000111101011100001010001111  
Add the 1+f parts, and call the result C:   
C = 10000000100_1.1101101101011100001010001111010111000010100011110101011  
Round the fraction part of C:  
C = 10000000100_1.1101101101011100001010001111010111000010100011110101  
Delete "1.":  
C = 10000000100___1101101101011100001010001111010111000010100011110101 

Convert the resultant float64 bit representation to decimal representation:

In [17]:
compose_float64("0", "10000000100", "1101101101011100001010001111010111000010100011110101")

59.419999999999995

# floating point multiplication
According to [https://www.sciencedirect.com/topics/computer-science/floating-point-addition](https://www.sciencedirect.com/topics/computer-science/floating-point-addition), the floating point multiplication rule is that

1. Add the biased exponents
2. Multiply the mantissas
3. Normalise
4. Round the result

### Explanation of 1.3 * 3.2 = 4.16

In [18]:
print(decompose_float64(1.3))
print(decompose_float64(3.2))

('0', '01111111111', '0100110011001100110011001100110011001100110011001101')
('0', '10000000000', '1001100110011001100110011001100110011001100110011010')


A = 01111111111___0100110011001100110011001100110011001100110011001101  
B = 10000000000___1001100110011001100110011001100110011001100110011010  
Prepend "1." to the fraction part:  
A = 01111111111_1.0100110011001100110011001100110011001100110011001101  
B = 10000000000_1.1001100110011001100110011001100110011001100110011010  
Add the biased exponent:  
01111111111 + 10000000000 - 01111111111 = 10000000000  
Multiply the 1+f parts:  
1.0100110011001100110011001100110011001100110011001101 * 1.1001100110011001100110011001100110011001100110011010 = 10.0001010001111010111000010100011110101110000101001000100001 (truncated to 58 bits after the binary point)  
Then C=A*B is:  
C = 10000000000_10.0001010001111010111000010100011110101110000101001000100001  
Normalize:  
C = 10000000001_1.00001010001111010111000010100011110101110000101001000100001  
Round:  
C = 10000000001_1.0000101000111101011100001010001111010111000010100100  

Remark:  
For multiplication of binaries, I used the function 'multiply_binary_strings' defined in the notebook 'example_4_1_loss_of_significance.ipynb'.  There is also a web service which performs binary calculation, such as 
[https://www.exploringbinary.com/binary-calculator](https://www.exploringbinary.com/binary-calculator) .

Convert the resultant float64 bit representation to decimal representation:

In [19]:
compose_float64("0", "10000000001", "0000101000111101011100001010001111010111000010100100")

4.16

### Explanation of 0.1 * 0.7 = 0.06999999999999999

In [20]:
print(decompose_float64(0.1))
print(decompose_float64(0.7))

('0', '01111111011', '1001100110011001100110011001100110011001100110011010')
('0', '01111111110', '0110011001100110011001100110011001100110011001100110')


A = 01111111011___1001100110011001100110011001100110011001100110011010  
B = 01111111110___0110011001100110011001100110011001100110011001100110  
Prepend "1." to the fraction part:  
A = 01111111011_1.1001100110011001100110011001100110011001100110011010  
B = 01111111110_1.0110011001100110011001100110011001100110011001100110  
Add the biased exponent:  
01111111011 + 01111111110 - 01111111111 = 01111111010  
Multiply the 1+f parts:  
1.1001100110011001100110011001100110011001100110011010 * 1.0110011001100110011001100110011001100110011001100110 = 10.0011110101110000101000111101011100001010001111010110111110 (truncated to 58 digits after the binary point)  
Then C=A*B is:  
C = 01111111010_10.0011110101110000101000111101011100001010001111010110111110    
Normalize:  
C = 01111111011_1.00011110101110000101000111101011100001010001111010110111110  
Round:  
C = 01111111011_1.0001111010111000010100011110101110000101000111101011    

Convert the resultant float64 bit representation to decimal representation:

In [21]:
compose_float64("0", "01111111011", "0001111010111000010100011110101110000101000111101011")

0.06999999999999999

### Explanation of 1e2 * 5.4321 = 543.21

In [22]:
print(decompose_float64(1e2))
print(decompose_float64(5.4321))

('0', '10000000101', '1001000000000000000000000000000000000000000000000000')
('0', '10000000001', '0101101110100111100001101100001000100110100000001010')


A = 10000000101___1001000000000000000000000000000000000000000000000000  
B = 10000000001___0101101110100111100001101100001000100110100000001010  
Prepend "1." to the fraction part:  
A = 10000000101_1.1001000000000000000000000000000000000000000000000000  
B = 10000000001_1.0101101110100111100001101100001000100110100000001010  
Add the biased exponent:  
10000000101 + 10000000001 - 01111111111 = 10000000111  
Multiply the 1+f parts:  
1.1001000000000000000000000000000000000000000000000000 * 1.0101101110100111100001101100001000100110100000001010 = 10.0001111100110101110000101000111101011100001010001111101  (truncated to 55 digits after the binary point)  
Then C=A*B is:  
C = 10000000111_10.0001111100110101110000101000111101011100001010001111101  
Normalize:  
C = 10000001000_1.00001111100110101110000101000111101011100001010001111101  
Round:  
C = 10000001000_1.0000111110011010111000010100011110101110000101001000

Thus, we see that the result is 543.21

In [23]:
compose_float64("0", "10000001000", "0000111110011010111000010100011110101110000101001000")

543.21